# 02 · Models and leave-one-season-out cross-validation
**UE24CS352A – Machine Learning · Mini-Project (Problem #101)** · **Team:** *<Name 1 – SRN>*, *<Name 2 – SRN>*

**Owner:** *<Name 2>* · **Reads:** `data/processed/bowl_matchups.csv` (run notebook 01 first) · **Writes:** `reports/results/cv_predictions.csv`, `model_comparison.csv`, `reports/figures/03–05`

Model code lives in `src/models.py`.

## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))          # lets us import the shared code in src/
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA_PROC, FIG_DIR, RES_DIR, SEED, PRIMARY_MODEL, ensure_dirs
from src.features import D_COLS
from src.models import MODELS, KIND, loso_cv, evaluate
try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50, "display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
np.random.seed(SEED)
ensure_dirs()
print("Project root:", ROOT)

In [ ]:
bowls = pd.read_csv(DATA_PROC / "bowl_matchups.csv", parse_dates=["date"])
print("Bowl games:", bowls.shape, "| games per season:", bowls.groupby("season").size().to_dict())

## 2. Models
All models share one interface — `fit(train_games)` and `predict(test_games)`. `predict` returns a **signed signal**
where `> 0` means "team A wins". Its magnitude is the confidence.

| Model | Type | Idea |
|---|---|---|
| Always pick listed home team | baseline | – |
| Higher regular-season win % | baseline | simple rule |
| Higher average margin | baseline | simple rule |
| **Score Regression** | regression | predict each team's points from its offence + opponent's defence (as in the paper) |
| Differential OLS (all features) | regression | predict margin from all differences |
| **Differential Ridge** *(primary)* | regression | same with L2 shrinkage, α chosen by inner CV |
| Differential OLS + p-value selection | regression | paper-style selection (p < 0.10), done **inside each training fold** |
| Logistic Regression | classifier | probability that A wins (L2, C by inner CV) |
| SVM (RBF) | classifier | decision function |
| Random Forest | classifier | win probability |

`PRIMARY_MODEL` is fixed *before* looking at results. The paper found differential regression best, and picking the best of
10 models on the same CV scores would give an optimistically biased estimate.

In [ ]:
print(len(MODELS), "models defined; primary model:", PRIMARY_MODEL)
print({k: v for k, v in KIND.items()})

## 3. Evaluation: leave-one-season-out cross-validation
Each season of bowl games is held out once and the model is trained on the other six, giving 7-fold CV. This mirrors the real use case
(predict *this* December from past data) and keeps all games of a season together.
Every data-dependent step (feature selection, scaling, hyper-parameter tuning) happens **inside** the training fold.

In [ ]:
cv = loso_cv(MODELS, bowls)
cv.to_csv(RES_DIR / "cv_predictions.csv", index=False)
print("CV predictions:", cv.shape)

### Pick'em simulation and metrics
For each season, rank games by the model's confidence (|signal|), give the least confident game 1 point and the most
confident N points, and score the points of the correct picks. A random picker earns about 50 % of the maximum.

In [ ]:
metrics = evaluate(cv)
metrics.to_csv(RES_DIR / "model_comparison.csv")
display(metrics.sort_values("pooled acc", ascending=False).round(3))

## 4. Results & analysis

In [ ]:
order = metrics.sort_values("pooled acc").index
fig, ax = plt.subplots(figsize=(8, 4.5))
acc = metrics.loc[order, "pooled acc"]
err = np.vstack([acc - metrics.loc[order, "95% CI low"], metrics.loc[order, "95% CI high"] - acc])
colors = ["#BAB0AC" if o.startswith("Baseline") else ("#E45756" if o == PRIMARY_MODEL else "#4C78A8") for o in order]
ax.barh(order, acc, xerr=err, color=colors, capsize=3)
ax.axvline(0.5, color="k", ls="--", lw=1); ax.set_xlim(0.35, 0.85)
ax.set_xlabel("Pooled leave-one-season-out accuracy (95% Wilson CI)")
ax.set_title("Bowl-game winner prediction")
plt.tight_layout(); plt.savefig(FIG_DIR / "03_model_comparison.png"); plt.show()

In [ ]:
show = [PRIMARY_MODEL, "Logistic Regression", "Score Regression", "Baseline: higher win %", "Baseline: listed home team"]
season_acc = cv[cv.model.isin(show)].groupby(["season", "model"]).correct.mean().unstack()[show]
fig, ax = plt.subplots(figsize=(8, 4))
for m in show:
    ax.plot(season_acc.index, season_acc[m], marker="o", lw=2.4 if m == PRIMARY_MODEL else 1.2, label=m)
ax.axhline(0.5, color="k", ls=":", lw=1)
ax.set_xlabel("Held-out season"); ax.set_ylabel("Pick accuracy"); ax.set_title("Accuracy by season")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(FIG_DIR / "04_accuracy_by_season.png"); plt.show()
display(season_acc.round(3))

### Is the confidence meaningful?
If |predicted margin| is a good confidence measure, accuracy should rise with it. This is the property needed for Pick'em.

In [ ]:
g = cv[cv.model == PRIMARY_MODEL].copy()
g["conf_bin"] = pd.qcut(g.conf, 3, labels=["low", "medium", "high"])
calib = g.groupby("conf_bin", observed=True).agg(games=("correct", "size"), accuracy=("correct", "mean"),
                                                  mean_abs_signal=("conf", "mean"))
display(calib.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(calib.index.astype(str), calib.accuracy, color="#4C78A8")
axes[0].axhline(0.5, color="k", ls="--", lw=1); axes[0].set_ylim(0.3, 1)
axes[0].set_title("Accuracy by confidence tercile"); axes[0].set_ylabel("Accuracy")
axes[1].scatter(g.signal, g.margin, s=14, alpha=0.6)
lim = max(abs(g.signal).max(), abs(g.margin).max())
axes[1].plot([-lim, lim], [-lim, lim], "k--", lw=1)
axes[1].set_xlabel("Predicted margin (CV)"); axes[1].set_ylabel("Actual margin")
axes[1].set_title(f"Predicted vs actual margin — {PRIMARY_MODEL}")
plt.tight_layout(); plt.savefig(FIG_DIR / "05_confidence_analysis.png"); plt.show()
print(f"Correlation(predicted, actual margin) = {np.corrcoef(g.signal, g.margin)[0, 1]:.3f}")

**Next:** open `03_final_model_and_demo.ipynb`.